In [1]:
from pathlib import Path

def find_project_root(start: Path, markers=("pyproject.toml", ".git")):
    for p in [start] + list(start.parents):
        if any((p / m).exists() for m in markers):
            return p
    raise RuntimeError("Project root not found")

ROOT = find_project_root(Path.cwd())

### nu01 (antistrophic)

Here we compute the contours and scores for the three first and final four lines of the famous entry of the Clouds that is used as example in the proceedings paper.

In [2]:
from lxml import etree
from responsio_accentuum import accentually_responding_syllables_of_strophes_polystrophic, count_all_accents_canticum

canticum = "nu01"

xml_file = ROOT / f"data/compiled/extra/test/test_nu01.xml"

tree = etree.parse(xml_file)
strophes = tree.xpath(f'//*[self::strophe or self::antistrophe][@responsion="{canticum}"]')
accent_maps = accentually_responding_syllables_of_strophes_polystrophic(*strophes)
all_accents = count_all_accents_canticum(tree, canticum, debug=True)

print("\n###################")
print("nu01 accent_maps:")
print(f"\tAcutes:")
for acute in accent_maps[0]:
    print(f"\t\t{acute}")
print(f"\tCircumflexes: {accent_maps[2]}")

print("\nnu01 all_accents: \n", all_accents)

canticum = "test01"

acute in syll 'έ' in 'nu01'.
acute in syll 'φέ' in 'nu01'.
circumflex in syll 'θῶ' in 'nu01'.
acute in syll 'φύ' in 'nu01'.
acute in syll 'ά' in 'nu01'.
circumflex in syll 'νοῦ' in 'nu01'.
acute in syll 'χέ' in 'nu01'.
acute in syll 'ρέ' in 'nu01'.
circumflex in syll 'γαῖς' in 'nu01'.
acute in syll 'σά' in 'nu01'.
acute in syll 'νέ' in 'nu01'.
acute in syll 'ς ὄμ' in 'nu01'.
acute in syll 'νά' in 'nu01'.
acute in syll 'δέ' in 'nu01'.
acute in syll 'δώ' in 'nu01'.
acute in syll 'κό' in 'nu01'.
acute in syll 'ὄμ' in 'nu01'.
circumflex in syll 'γαῖ' in 'nu01'.
acute in syll 'θέ' in 'nu01'.
acute in syll 'φό' in 'nu01'.
acute in syll 'ἔλ' in 'nu01'.
acute in syll 'χθό' in 'nu01'.
acute in syll 'λά' in 'nu01'.
acute in syll 'ς, εὔ' in 'nu01'.
circumflex in syll 'γᾶν' in 'nu01'.
acute in syll 'Κέκ' in 'nu01'.
acute in syll 'ό' in 'nu01'.
acute in syll 'ή' in 'nu01'.
circumflex in syll 'παῖ' in 'nu01'.
acute in syll 'ν ὥ' in 'nu01'.
circumflex in syll 'ἦ' in 'nu01'.
acute in syll 'ρί' in 'nu0

### ach01 (2-strophic)

Naturally we need guaranteed true orthographic accent responsion stats for at least one antistrophic and one polystrophic song to know that the stats code works.

I chose ach01 and ach05.

- ach01: 
  - Accentual responsion (confirms visualization.py): 
    - Acute:      
      - 0 + 3 + 0 + 2 + 0 + 1 + 0 + 1 + 2 = 9
    - Grave:      
      - 1 + 0 + 0 + 1 + 0 + 0 + 0 + 0 + 0 = 2
    - Circumflex: 
      - 1 + 0 + 0 + 0 + 0 + 1 + 1 + 0 + 1 = 4

My manual check confirms the accent map of `accentually_responding_syllables_of_strophes_polystrophic`.

Using `count_all_accents_canticum(tree, canticum)` we can get the total number of acutes in both strophes as 62, which I have manually confirmed by reckoning in the file `sanity_check/ach01.tsv`. Before we compute the fraction we have two count every responding syllable twice, because it actually involves two accents. Finally, this means we have a confirmed statistic of **9*2/62 ≈ 0.29**.


In [3]:
accent_maps = [
    [
        {('205', 7): 'πάν', ('220', 7): 'τεί'}, 
        {('205', 10): 'πό', ('220', 10): 'κέ'}, 
        {('205', 13): 'ἄξ', ('220', 13): 'ρύ'}, 
        {('207', 4): "δ' ὅ", ('222', 4): 'γέ'}, 
        {('207', 14): 'φέ', ('222', 14): 'νέ'}, 
        {('210-211', 3): 'ς. Οἴ', ('225', 3): 'οί'}, 
        {('213-214', 5): 'τί', ('227', 5): 'ρί'}, 
        {('215-217', 3): 'λού', ('230-233', 3): 'νή'}, 
        {('215-217', 23): 'τό', ('230-233', 23): 'ρό'}
        ], 
    [
        {('204', 10): 'τὸ', ('219', 10): 'μὸ'}, 
        {('207', 11): 'τὰς', ('222', 11): 'γὼ'}
        ],
    [
        {('204', 1): 'Τῇ', ('219', 1): 'Νῦν'}, 
        {('210-211', 7): 'τῶν', ('225', 7): 'θροῖ'}, 
        {('211-212', 4): 'μῆς', ('226', 4): 'μοῦ'}, 
        {('215-217', 10): 'ὧ', ('230-233', 10): 'τοῖ'}
        ]
    ]

The easiest way to manually count is to print the sylls from all responding lines aligned in tabs: 

In [4]:
from lxml import etree as ET

def extract_responsion_syllables_vertical(filename: str, responsion_value: str) -> str:
    tree = ET.parse(filename)
    root = tree.getroot()

    # Collect all <strophe> and <antistrophe> with matching responsion
    matched_blocks = []
    for tag in ('strophe', 'antistrophe'):
        matched_blocks.extend(root.xpath(f".//{tag}[@responsion='{responsion_value}']"))

    # Extract <l> elements from each matched block
    all_l_lists = [block.findall("l") for block in matched_blocks]
    max_lines = max(len(l_list) for l_list in all_l_lists)

    output_lines = []

    for line_index in range(max_lines):
        for l_list in all_l_lists:
            if line_index < len(l_list):
                l_elem = l_list[line_index]

                # Get all <syll> elements (including those in <conjecture>)
                syll_elems = l_elem.xpath(".//syll")

                merged_sylls = []
                i = 0
                while i < len(syll_elems):
                    syll = syll_elems[i]
                    text = (syll.text or "").strip()
                    if i + 1 < len(syll_elems):
                        next_syll = syll_elems[i + 1]
                        if (syll.get("resolution") == "True" and
                            next_syll.get("resolution") == "True"):
                            merged = text + (next_syll.text or "").strip()
                            merged_sylls.append(merged)
                            i += 2
                            continue
                    merged_sylls.append(text)
                    i += 1

                output_lines.append("\t".join(merged_sylls))
            else:
                output_lines.append("")  # No line at this index in this block
        output_lines.append("")  # Blank line between line groups

    return "\n".join(output_lines)

canticum = "ach01"
infix = canticum[:-2]

input_file = ROOT / f"data/compiled/extra/test/test_ach.xml"

output_lines = extract_responsion_syllables_vertical(input_file, canticum)

print(output_lines)
with open(ROOT / f"tests/{canticum}.tsv", "w") as f:
    f.write(output_lines)

Τῇ	δε	πᾶ	ς ἕ	που	δί	ω	κε	καὶ	τὸ	ν ἄν	δρα	πυν	θά	νου
Νῦν	δ' ἐ	πει	δὴ	στερ	ρὸ	ν ἤ	δη	τοὐ	μὸ	ν ἀν	τικ	νή	μι	ον

τῶν	ὁ	δοι	πό	ρων	ἁ	πάν	των	τῇ	πό	λει	γὰρ	ἄξ	ι	ον
καὶ	πα	λαι	ῷ	Λακ	ρα	τεί	δῃ	τὸ σ	κέ	λος	βα	ρύ	νε	ται

ξυλ	λα	βεῖν	τὸν	ἄν	δρα	τοῦ	τον	Ἀλ	λά	μοι	μη	νύ	σα	τε
οἴ	χε	ται	Δι	ωκ	τέ	ος	δέ	μὴ	γὰ	ρ ἐγ	χά	νοι	πο	τὲ

εἴ	τις	οἶ	δ' ὅ	ποι	τέ	τραπ	ται	γῆς	ὁ	τὰς	σπον	δὰς	φέ	ρων
μη	δέ	περ	γέ	ρον	τα	ς ὄν	τα	ς ἐκ	φυ	γὼ	ν Ἀ	χαρ	νέ	ας

Ἐκ	πέ	φευγ	οἴ	χε	ται
ὅσ	τι	ς, ὦ Ζ	εῦ	πά	τερ

φροῦ	δο	ς. Οἴ	μοι	τά	λας	τῶν	ἐ	τῶν	τῶν	ἐ	μῶν
καὶ	θε	οί	τοῖ	σι	ν ἐχ	θροῖ	σιν	ἐσ	πεί	σα	το

οὐκ	ἂν	ἐπ' ἐ	μῆς	γε	νεό	τη	το	ς, ὅτ' ἐ	γὼ	φέ	ρων
οἷ	σι	παρἐ	μοῦ	πό	λεμο	ς ἐχ	θο	δοπὸ	ς αὔξ	ε	ται

ἀν	θρά	κων	φορ	τί	ον
τῶ	ν ἐ	μῶν	χω	ρί	ων

ἠ	κο	λού	θουν	Φα	ΰλ	λῳ τ	ρέ	χων	ὧ	δε	φαύ	λως	ἂ	νὁ σ	πον	δο	φόρο	ς οὗ	το	ς ὑπ' ἐ	μοῦ	τό	τεδι	ω	κό	μενο	ς ἐξ	έ	φυγε	ν οὐ	δ' ἂν	ἐλα	φρῶ	ς ἂ	ν ἀπε	πλίξ	α	το
κοὐ	κ ἀ	νή	σω π	ρὶ	ν ἂν	σχοῖ	νο	ς αὐ	τοῖ	σι	ν ἀν	τεμ	πα	γῶ	καὶ σ	κό	λοψ	ὀξ	ύ	ς, ὀδυ	νη	ρό	ς, ἐπί	κω	πο	ς, ἵνα	μή	πο	τεπα	τῶ	σι	ν ἔτι	τὰ	ς ἐ	μὰ	ς ἀμ

### Polystrophic: ach05

`accentually_responding_syllables_of_strophes_polystrophic` yields an empty accent_map for the quartet `ach05`. We can convince ourselves that this is not simply a lack of support for polystrophy by creating a version with an artificial responsion in the first syllable:

In [5]:
from lxml import etree
from responsio_accentuum import accentually_responding_syllables_of_strophes_polystrophic, count_all_accents_canticum

canticum = "ach05"

xml_file = ROOT / f"data/compiled/extra/test/test_ach.xml"

tree = etree.parse(xml_file)
strophes = tree.xpath(f'//*[self::strophe or self::antistrophe][@responsion="{canticum}"]')
accent_maps = accentually_responding_syllables_of_strophes_polystrophic(*strophes)

print("ach05 accent_maps: ", accent_maps)

canticum = "test01"

xml_file = ROOT / f"data/compiled/extra/test/mockup_polystrophic.xml"

tree = etree.parse(xml_file)
strophes = tree.xpath(f'//*[self::strophe or self::antistrophe][@responsion="{canticum}"]')
accent_maps = accentually_responding_syllables_of_strophes_polystrophic(*strophes)
all_accents = count_all_accents_canticum(tree, canticum)

print("test01 accent_maps: ", accent_maps)
print("all_accents: ", all_accents)

ach05 accent_maps:  [[], [], []]
test01 accent_maps:  [[], [], [{('836-837', 1): 'χλαῖ', ('841-842', 1): 'χλαῖ', ('847-848', 1): 'χλαῖ', ('853', 1): 'χλαῖ'}]]
all_accents:  {'acute': 43, 'grave': 17, 'circumflex': 30}
